In [0]:
# Read from the Bronze Delta Lake table
df_cast = (
    spark.read
    .format("delta")
    .load("abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_cast")
)


In [0]:
import pyspark.sql.functions as F

# 1. Check Row Counts
raw_df_cast = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("delimiter", ",")
    .csv("abfss://raw@mynetflixproject.dfs.core.windows.net/netflix_cast/")
)
raw_count = raw_df_cast.count()
bronze_count = df_cast.count()
count_difference = bronze_count - raw_count

# 2. Standardise schemas to strings for an exact text content comparison
raw_standardised = raw_df_cast.select([F.col(c).cast("string").alias(c) for c in raw_df_cast.columns])
bronze_standardised = df_cast.select([F.col(c).cast("string").alias(c) for c in raw_df_cast.columns])

# 3. Check for specific content differences
rows_in_raw_not_bronze = raw_standardised.subtract(bronze_standardised).count()
rows_in_bronze_not_raw = bronze_standardised.subtract(raw_standardised).count()

# 4. Print Complete Validation Summary
print("=" * 45)
print("🔍 COMPLETE RAW VS. BRONZE VERIFICATION")
print("=" * 45)
print(f"Raw Row Count:                {raw_count:,}")
print(f"Bronze Row Count:             {bronze_count:,}")
print(f"Row Count Difference:         {count_difference:,}")
print("-" * 45)
print(f"Rows in Raw but NOT Bronze:   {rows_in_raw_not_bronze}")
print(f"Rows in Bronze but NOT Raw:   {rows_in_bronze_not_raw}")
print("=" * 45)

# Final Verdict Check
counts_match = (raw_count == bronze_count)
content_matches = (rows_in_raw_not_bronze == 0 and rows_in_bronze_not_raw == 0)

if counts_match and content_matches:
    print("✅ SUCCESS: Data is 100% identical in both count and content!")
else:
    print("❌ ERROR: Mismatch detected! Check the numbers above.")
print("=" * 45)


In [0]:
import pyspark.sql.functions as F

# 1. Check Row Counts
raw_df_cast = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("delimiter", ",")
    .csv("abfss://raw@mynetflixproject.dfs.core.windows.net/netflix_cast/")
)
raw_count = raw_df_cast.count()
bronze_count = df_cast.count()
count_difference = bronze_count - raw_count

# 2. Standardise schemas to strings for an exact text content comparison
raw_standardised = raw_df_cast.select([F.col(c).cast("string").alias(c) for c in raw_df_cast.columns])
bronze_standardised = df_cast.select([F.col(c).cast("string").alias(c) for c in raw_df_cast.columns])

# 3. Check for specific content differences
rows_in_raw_not_bronze = raw_standardised.subtract(bronze_standardised).count()
rows_in_bronze_not_raw = bronze_standardised.subtract(raw_standardised).count()

# 4. Print Complete Validation Summary
print("=" * 45)
print("🔍 COMPLETE RAW VS. BRONZE VERIFICATION")
print("=" * 45)
print(f"Raw Row Count:                {raw_count:,}")
print(f"Bronze Row Count:             {bronze_count:,}")
print(f"Row Count Difference:         {count_difference:,}")
print("-" * 45)
print(f"Rows in Raw but NOT Bronze:   {rows_in_raw_not_bronze}")
print(f"Rows in Bronze but NOT Raw:   {rows_in_bronze_not_raw}")
print("=" * 45)

# Final Verdict Check
counts_match = (raw_count == bronze_count)
content_matches = (rows_in_raw_not_bronze == 0 and rows_in_bronze_not_raw == 0)

if counts_match and content_matches:
    print("✅ SUCCESS: Data is 100% identical in both count and content!")
else:
    print("❌ ERROR: Mismatch detected! Check the numbers above.")
print("=" * 45)


In [0]:
from pyspark.sql import functions as F

df_cast = spark.read.format("delta").load(
    "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_cast"
)

cast_col = F.col("cast")
cast_trimmed = F.trim(cast_col)

# 1. Single-pass summary of text quality issues in 'cast'
df_cast.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(cast_trimmed).alias("distinct_actors"),
    F.sum(F.when(cast_col.isNull() | (cast_trimmed == ""), 1).otherwise(0)).alias("dq_null_or_blank"),
    F.sum(F.when(cast_col != cast_trimmed, 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(cast_col.rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    F.sum(F.when(cast_col.contains(","), 1).otherwise(0)).alias("dq_contains_comma"),
    F.sum(F.when(cast_trimmed.rlike(r"^[0-9\W_]+$"), 1).otherwise(0)).alias("dq_numeric_or_symbol_only"),
    F.sum(F.when(F.lower(cast_trimmed).isin("unknown", "n/a", "na", "none", "null", "tbd"), 1).otherwise(0)).alias("dq_placeholder_text")
).display()

# 2. Inspect any suspicious rows (commas, numeric/symbols, placeholders, or very short names <= 3 chars)
(
    df_cast
    .filter(
        cast_col.isNull() | (cast_trimmed == "") |
        cast_col.contains(",") |
        cast_trimmed.rlike(r"^[0-9\W_]+$") |
        F.lower(cast_trimmed).isin("unknown", "n/a", "na", "none", "null", "tbd") |
        (F.length(cast_trimmed) <= 3)
    )
    .select("show_id", "cast", F.length(cast_trimmed).alias("char_len"))
    .orderBy("char_len", "cast")
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Clean whitespace (trim outer spaces, collapse repeated internal spaces, blank -> NULL)
staged_cast_df = df_cast.withColumn(
    "cast",
    F.when(
        F.col("cast").isNull() | (F.trim(F.col("cast")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.trim(F.regexp_replace(F.col("cast"), r"\s+", " "))
    )
)

# 2. Quarantine invalid cast values (NULL, blank, or numeric/symbol-only junk; preserves 1-char stage names like 'L')
is_invalid_cast = (
    F.col("cast").isNull() |
    F.col("cast").rlike(r"^[0-9\W_]+$")
)

quarantine_cast_col_df = staged_cast_df.filter(is_invalid_cast)
clean_cast_df = staged_cast_df.filter(~is_invalid_cast)

# 3. Post-transformation validation check for 'cast'
clean_cast_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("cast").alias("distinct_actors"),
    F.sum(F.when(F.col("cast").isNull() | (F.trim(F.col("cast")) == ""), 1).otherwise(0)).alias("val_null_or_blank"),
    F.sum(F.when(F.col("cast") != F.trim(F.col("cast")), 1).otherwise(0)).alias("val_outer_spaces"),
    F.sum(F.when(F.col("cast").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces")
).display()


xx


In [0]:
from pyspark.sql import functions as F

silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

id_str = F.trim(F.col("show_id").cast("string"))

# 1. Audit show_id quality AND composite duplicate (show_id, cast) pairs
clean_cast_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(id_str.isNull() | (id_str == ""), 1).otherwise(0)).alias("dq_null_show_id"),
    F.sum(F.when(id_str.isNotNull() & ~id_str.rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("dq_non_digit_show_id"),
    F.countDistinct(id_str).alias("distinct_show_ids"),
    (F.count("*") - F.countDistinct(id_str, F.lower(F.col("cast")))).alias("dq_duplicate_pairs")
).display()

# 2. Check for orphan show_ids missing from Silver netflix_titles
orphan_cast_df = (
    clean_cast_df
    .withColumn("show_id_clean", id_str)
    .join(silver_titles_ids, F.col("show_id_clean") == silver_titles_ids.show_id, "left_anti")
)

print(f"Orphan show_id rows in netflix_cast: {orphan_cast_df.count()}")
orphan_cast_df.display()


In [0]:
from pyspark.sql import functions as F

silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

# 1. Trim whitespace and cast show_id cleanly to STRING
staged_id_df = clean_cast_df.withColumn("show_id", F.trim(F.col("show_id").cast("string")))

# 2. Check foreign-key match against Silver netflix_titles
staged_with_fk = (
    staged_id_df.alias("c")
    .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
    .select(
        F.col("c.*"),
        F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
    )
)

# 3. Quarantine rows with missing, non-digit, or orphan show_id
is_invalid_show_id = (
    F.col("show_id").isNull() |
    (F.col("show_id") == "") |
    ~F.col("show_id").rlike(r"^[0-9]+$") |
    F.col("_is_orphan")
)

quarantine_cast_id_df = (
    staged_with_fk
    .filter(is_invalid_show_id)
    .withColumn(
        "quarantine_reason",
        F.when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
         .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
         .otherwise("Unknown show_id error")
    )
    .drop("_is_orphan")
)

# 4. Keep valid show_id rows as STRING and deduplicate composite key (show_id, cast)
clean_cast_df = (
    staged_with_fk
    .filter(~is_invalid_show_id)
    .drop("_is_orphan")
    .dropDuplicates(["show_id", "cast"])
)


In [0]:
from pyspark.sql import functions as F

clean_cast_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("show_id").alias("distinct_show_ids"),
    F.countDistinct("show_id", "cast").alias("distinct_pairs"),
    F.sum(F.when(F.col("show_id").isNull() | ~F.col("show_id").rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("val_invalid_show_ids")
).display()

print(f"Quarantined show_id rows: {quarantine_cast_id_df.count()}")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_cast"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_cast"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_cast"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_cast"

def process_cast_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    # Read valid parent show_ids from Silver netflix_titles for foreign-key check
    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Standardise show_id (trimmed STRING) and cast (collapse double spaces, trim outer spaces)
    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "cast",
            F.when(F.col("cast").isNull() | (F.trim(F.col("cast")) == ""), F.lit(None).cast("string"))
             .otherwise(F.trim(F.regexp_replace(F.col("cast"), r"\s+", " ")))
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("c")
        .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("c.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Unified quarantine condition (preserves 1-char stage names like 'L')
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("cast").isNull() | F.col("cast").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk
        .filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("cast").isNull() | F.col("cast").rlike(r"^[0-9\W_]+$"), "Invalid or Missing cast")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk
        .filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "cast"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.cast = source.cast"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

# Stream ONLY new Bronze commits using Structured Streaming
query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_cast_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_cast.")


In [0]:
silver_cnt     = spark.read.format("delta").load(silver_path).count()
quarantine_cnt = spark.read.format("delta").load(quarantine_path).count()

print(f"Silver netflix_cast rows:     {silver_cnt:,}")
print(f"Quarantine netflix_cast rows: {quarantine_cnt:,}")
